In [1]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-5-5"

In [2]:
# ヘルパー関数(L26 の内容。提供済み。temperature は現行モデルで使えないため外している)
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, tools=None):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    return client.messages.create(**params)  # テキストだけでなく、Message 全体を返す


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

## 課題: 複数のツールを、会話ループで回す

- 対応: L26「ツールを使うマルチターン会話」、L27「複数ターンの実装」、L28「複数のツールを使用する」。
- 今回、新しく覚えること: **ツール使用の会話ループ**(Claude がツールを要求しなくなるまで、実行して結果を返す)。
- 目標: 「医者の予約のリマインダーを設定してください。2050年1月1日から177日後です。」に対して、Claude が `add_duration_to_datetime` → `set_reminder` の順にツールを呼び、最終応答を返すまでを、ループで自動処理する。
- 提供済み: ヘルパー関数、3つのツールの関数とスキーマ(コース側の教材をもとにした)。穴埋めは **TODO 1〜3** の3つ。TODO 4 は実行と観察。
- 流れ: TODO 1(`run_tool`)→ TODO 2(`run_tools`)→ TODO 3(`run_conversation`)→ TODO 4(実行して観察)。TODO 1・2 は、API を呼ばずに確認できる。
- API 呼び出し: TODO 4 で、数回(少額の料金がかかる)。
- 注意: コースの参考ノートブック(完成コード入り)は、**開かずに**進める。詰まったら、下の「ヒント」のセルを見る。
- 注意: 実行の順番は、上から。やり直すときは、`messages = []` のセルから。

In [3]:
# ツールの関数とスキーマ(提供済み。TODO ではない)
from datetime import datetime, timedelta
from anthropic.types import ToolParam


def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)


def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        days_in_month = [
            31,
            29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
            31, 30, 31, 30, 31, 31, 30, 31, 30, 31,
        ]
        day = min(date.day, days_in_month[month - 1])
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")


get_current_datetime_schema = ToolParam(
    {
        "name": "get_current_datetime",
        "description": "Returns the current date and time formatted according to the specified format string. Use this tool when you need to know the current date and time. The default format returns the date and time in ISO-like format (YYYY-MM-DD HH:MM:SS).",
        "input_schema": {
            "type": "object",
            "properties": {
                "date_format": {
                    "type": "string",
                    "description": "A string specifying the format of the returned datetime. Uses Python's strftime format codes. The default is '%Y-%m-%d %H:%M:%S'.",
                }
            },
            "required": [],
        },
    }
)

add_duration_to_datetime_schema = ToolParam(
    {
        "name": "add_duration_to_datetime",
        "description": "Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format (e.g., 'Thursday, April 03, 2025 10:30:00 AM'). Handles seconds, minutes, hours, days, weeks, months, and years, including month lengths and leap years. Use this tool whenever a date calculation is needed.",
        "input_schema": {
            "type": "object",
            "properties": {
                "datetime_str": {
                    "type": "string",
                    "description": "The input datetime string to which the duration will be added. Must match input_format.",
                },
                "duration": {
                    "type": "number",
                    "description": "The amount of time to add. Can be negative for past dates. Defaults to 0.",
                },
                "unit": {
                    "type": "string",
                    "description": "The unit of time. Must be one of: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', or 'years'. Defaults to 'days'.",
                },
                "input_format": {
                    "type": "string",
                    "description": "The strptime format for datetime_str. Defaults to '%Y-%m-%d'.",
                },
            },
            "required": ["datetime_str"],
        },
    }
)

set_reminder_schema = ToolParam(
    {
        "name": "set_reminder",
        "description": "Creates a timed reminder that will notify the user at the specified time with the provided content. Use this tool when a user wants to be reminded about something at a future point in time.",
        "input_schema": {
            "type": "object",
            "properties": {
                "content": {
                    "type": "string",
                    "description": "The message text shown in the reminder notification, such as 'Take medication'.",
                },
                "timestamp": {
                    "type": "string",
                    "description": "The date and time when the reminder should trigger, as an ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS).",
                },
            },
            "required": ["content", "timestamp"],
        },
    }
)

TOOLS = [get_current_datetime_schema, add_duration_to_datetime_schema, set_reminder_schema]

In [4]:
# TODO 1: ツールのルーティング run_tool を完成させる(L27・L28)
#   - tool_name に応じて、対応する関数を **tool_input で呼び、結果を返す。
#   - 対象は3つ: get_current_datetime / add_duration_to_datetime / set_reminder。
#   - どれにも一致しないツール名は、ValueError を発生させる(メッセージに、受け取ったツール名を入れる)。
def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    # TODO: add_duration_to_datetime の分岐
    if tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    # TODO: set_reminder の分岐
    if tool_name == "set_reminder":
        return set_reminder(**tool_input)
    # TODO: どれにも一致しない場合は ValueError
    raise ValueError(f"Unknown tool name: {tool_name}")

In [5]:
# TODO 1 の動作確認(API は呼ばない)。すべて通れば、"OK" が表示される。
assert len(run_tool("get_current_datetime", {"date_format": "%H:%M"})) == 5
assert "June 27, 2050" in run_tool(
    "add_duration_to_datetime", {"datetime_str": "2050-01-01", "duration": 177, "unit": "days"}
)

try:
    run_tool("no_such_tool", {})
    raise AssertionError("未登録のツール名で、ValueError が出ていない")
except ValueError as e:
    assert "no_such_tool" in str(e), "メッセージに、ツール名が入っていない"
    print("エラーメッセージ:", e)

print("OK")

エラーメッセージ: Unknown tool name: no_such_tool
OK


In [18]:
# TODO 2: run_tools を完成させる(L27)
#   - message.content から、type が "tool_use" のブロックだけを集める(リスト内包表記)。
#   - 各ブロックについて run_tool を実行し、tool_result ブロック(辞書)を作ってリストに足す。
#       成功: {"type": "tool_result", "tool_use_id": ブロックの id, "content": json.dumps(出力), "is_error": False}
#       失敗: 例外を except で受け、content に f"Error: {e}"、is_error を True にした tool_result を作る。
#   - 結果ブロックのリストを返す(ページのコードは、この return が無い)。
import json


def run_tools(message):
    tool_requests = [block for block in message.content if block.type == "tool_use"]  # TODO: type が "tool_use" のブロックだけを集める
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)  # TODO: run_tool に、ブロックの name と input を渡す
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,  # TODO: ブロックの id
                "content": json.dumps(tool_output),
                "is_error": False,
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,  # TODO: ブロックの id
                "content": f"Error: {e}",  # TODO: f"Error: {e}" の形
                "is_error": True,  # TODO: エラーなので…
            }

        tool_result_blocks.append(tool_result_block)

    # TODO: 結果ブロックのリストを返す
    return tool_result_blocks

In [19]:
# TODO 2 の動作確認(API は呼ばない。偽の応答を作って確認する)。すべて通れば、"OK" が表示される。
from types import SimpleNamespace as NS

fake = NS(content=[
    NS(type="text", text="計算します"),
    NS(type="tool_use", id="toolu_A", name="add_duration_to_datetime",
       input={"datetime_str": "2050-01-01", "duration": 177, "unit": "days"}),
    NS(type="tool_use", id="toolu_B", name="add_duration_to_datetime",
       input={"datetime_str": "2050-01-01", "duration": 2, "unit": "fortnights"}),
    NS(type="tool_use", id="toolu_C", name="no_such_tool", input={}),
])

results = run_tools(fake)
print(results[1])
assert results is not None and len(results) == 3, "結果ブロックは、ToolUse の数(3つ)になる。テキストブロックは含めない"

assert [r["tool_use_id"] for r in results] == ["toolu_A", "toolu_B", "toolu_C"], "tool_use_id が、依頼の id と対応していない"
assert results[0]["is_error"] is False and "June 27, 2050" in results[0]["content"]
assert results[1]["is_error"] is True and results[1]["content"].startswith("Error:")
assert results[2]["is_error"] is True and "no_such_tool" in results[2]["content"]
assert all(r["type"] == "tool_result" for r in results)

print("fortnights の結果:", results[1]["content"])
print("OK")

{'type': 'tool_result', 'tool_use_id': 'toolu_B', 'content': 'Error: Unsupported time unit: fortnights', 'is_error': True}
fortnights の結果: Error: Unsupported time unit: fortnights
OK


In [20]:
# TODO 3: 会話ループ run_conversation を完成させる(L26・L27)
#   - chat を、tools=TOOLS 付きで呼ぶ。
#   - 応答を、アシスタントメッセージとして履歴に足す(response をそのまま渡す)。
#   - 応答のテキストを表示する(text_from_message)。
#   - stop_reason が "tool_use" でなければ、終了して messages を返す。
#   - "tool_use" なら run_tools で結果を作り、ユーザーメッセージとして履歴に足して、次の周回へ。
#   - 安全装置: max_turns 回まわっても終わらなければ、RuntimeError を発生させる(ページには無い、こちらの設計)。
def run_conversation(messages, max_turns=10):
    for turn in range(max_turns):
        response = chat(messages, tools=TOOLS)  # TODO: tools=TOOLS を付けて chat を呼ぶ

        # TODO: 応答を、アシスタントメッセージとして履歴に足す
        add_assistant_message(messages, response)
        # TODO: 応答のテキストを表示する
        print(text_from_message(response))

        # TODO: stop_reason が "tool_use" でなければ、messages を返す
        if response.stop_reason != "tool_use":
            return messages
        # TODO: ツールを実行して結果を作り、ユーザーメッセージとして履歴に足す
        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

    raise RuntimeError(f"{max_turns} 回まわっても、終わらなかった")

## TODO 4: 実行して、観察する(API を呼ぶ)

下の2つのセルを実行する。1つ目は、ページと同じ依頼。2つ目は、履歴の中身を、種類ごとに並べて表示する(提供済み)。

In [21]:
# TODO 4-a: 実行する(コードは、そのまま)
messages = []
add_user_message(
    messages,
    "医者の予約のリマインダーを設定してください。2050年1月1日から177日後です。",
)
run_conversation(messages)



----
Setting the following reminder for 2050-06-27T09:00:00:
医者の予約
----
2050年1月1日から177日後は **2050年6月27日（月）** です。この日の午前9時に「医者の予約」のリマインダーを設定しました。

予約の時間が決まっている場合は教えてください。その時刻に合わせて設定し直します。


[{'role': 'user', 'content': '医者の予約のリマインダーを設定してください。2050年1月1日から177日後です。'},
 {'role': 'assistant',
  'content': [ThinkingBlock(signature='CAQS+ggKEAgTGAI4AUIIdGhpbmtpbmcSDPto5+IBM15vuN/jmhoMfxdnA5g63z6VV0NsIjBeaNDWT3tp97+F37FbUzBVEB4iOydW/otnC98p81O6Xq8cFscnwVLFL4DsoiLAmBEqlwjvVzvFzcUD1NUw4ugvOmEBUT9ZazK6/hc3JrPnc5uPDhizE5B2+UbUzYjcIBAornoyT27fHNrE17cetjn6XJnAaUWRRhMfvRYGeDl3dxIVgvcob9N8jaxOMHLEkweRCHvZgg2v/obqGjUnYiNiRK2kXMrYl2td2AKuyOK9pH7BtUAVrKBaYxYDbDCuJjvbKrNwtQ1+HMjgMtVQLNnvrqf5ihf4vmCzD7PD1dhhyWQKI5RaQ1zba0dMsI7Ndi2q6BOfeIA8/D66VWuAKSX5GuKBL2SenDhVauaDnJpdk05HLddVw20eEcE97/Xe4Q2+hLmCvgRC3EMYIugTSoFacm8jdqQsYJyDgXR7gDgJCRqHm7XUk7bXNCSNSbCTjjpcsRtiJpcm8z0TGrhfYAHlfPy2yZAkttdtQYuaACyTs5wV+8X1ZZrm6RcL9MsZkt9698fwy8fH+VkHWWTI31vGd7xL9F5oy97l51rrEIcIPl+Z1nvEY5O5B8TBZ8ZRt6e6VYnyZv1pypJC0W7QH5KcggNzBSgTbPpeygFhEM0ooMiZMRfLmpSIMzTJZC/zY7IKZ5OrI1dq5SvZCWKV6gAFcuWLAI6WMs9upvgFJ6hH2+O2K4Qmm7MMUkp5w+2ShQ3VDGU4ShrSb9VzOowrVgKOaLMQ7idozDkEka2HHPSsCK05RkiEesxdSJxUWju/T4/p/I86dX1eHukg+JSfZBJ4rqz

In [23]:
# TODO 4-b: 履歴を、種類ごとに表示する(提供済み)
def show_history(messages):
    for i, m in enumerate(messages):
        content = m["content"]
        if isinstance(content, str):
            print(f"[{i}] {m['role']}: (文字列) {content[:60]}")
            continue
        for b in content:
            t = b["type"] if isinstance(b, dict) else b.type
            if t == "tool_use":
                print(f"[{i}] {m['role']}: tool_use  name={b.name} input={b.input}")
            elif t == "tool_result":
                print(f"[{i}] {m['role']}: tool_result is_error={b['is_error']} content={b['content']}")
            elif t == "text":
                print(f"[{i}] {m['role']}: text  {b.text[:60]}")


show_history(messages)

[0] user: (文字列) 医者の予約のリマインダーを設定してください。2050年1月1日から177日後です。
[1] assistant: tool_use  name=add_duration_to_datetime input={'datetime_str': '2050-01-01', 'duration': 177, 'unit': 'days', 'input_format': '%Y-%m-%d'}
[2] user: tool_result is_error=False content="Monday, June 27, 2050 12:00:00 AM"
[3] assistant: tool_use  name=set_reminder input={'content': '医者の予約', 'timestamp': '2050-06-27T09:00:00'}
[4] user: tool_result is_error=False content=null
[5] assistant: text  2050年1月1日から177日後は **2050年6月27日（月）** です。この日の午前9時に「医者の予約」のリマイン


In [24]:
# TODO 4-c: エラーを起こして、Claude の動きを見る
#   - 未対応の単位を使わせる依頼。Claude が自分で weeks などに直す場合もある(その場合は、エラーにならない)。
#   - エラーにならなかったときは、依頼の文を変えて、もう一度試してよい。
messages = []
add_user_message(
    messages,
    "add_duration_to_datetime ツールを、unit に 'fortnights'、duration に 2 を指定して呼び出し、2050-01-01 に加算してください。",
)
run_conversation(messages)
show_history(messages)


ツールがエラーを返しました。`fortnights`(フォートナイト)は、このツールで指定できる単位ではありませんでした。

```
Error: Unsupported time unit: fortnights
```

このツールで使える単位は `seconds`、`minutes`、`hours`、`days`、`weeks`、`months`、`years` です。2 フォートナイトは 4 週間と同じなので、unit を `weeks`、duration を 4 にして 2050-01-01 に加算することもできます。その計算を行いましょうか。
[0] user: (文字列) add_duration_to_datetime ツールを、unit に 'fortnights'、duration に
[1] assistant: tool_use  name=add_duration_to_datetime input={'datetime_str': '2050-01-01', 'duration': 2, 'unit': 'fortnights'}
[2] user: tool_result is_error=True content=Error: Unsupported time unit: fortnights
[3] assistant: text  ツールがエラーを返しました。`fortnights`(フォートナイト)は、このツールで指定できる単位ではありませんでした


## 観察の質問(ノートブックには書かず、チャットで答える)

1. TODO 4-a で、`stop_reason` が `"tool_use"` になった回数と、`"end_turn"` になった回数は、それぞれ何回か。ツールが呼ばれた順番は。
2. `set_reminder` の `tool_result` の `content` は何だったか。関数は何も返していないのに、Claude はそれを成功と受け取ったか。この設計を、どう直すか。
3. TODO 4-c で、`is_error: True` を受けた Claude は、次に何をしたか(自分で直して再試行した / 利用者に説明した / その他)。
4. 安全装置 `max_turns` は、なぜ必要か。上限が無いと、どんな場面で困るか。

## ヒント(詰まったときだけ見る)

- **TODO 1**: ex11 の TODO と同じ形。`elif tool_name == "...":` を2つ足し、最後に `else:` で `raise ValueError(f"... {tool_name}")` を書く。
- **TODO 2**:
  - 絞り込みは `[block for block in message.content if block.type == "tool_use"]`。
  - ブロックの属性は `tool_request.name`、`tool_request.input`、`tool_request.id`。
  - `except` 側の `content` は `f"Error: {e}"`、`is_error` は `True`。
  - 最後の行は `return tool_result_blocks`。
- **TODO 3**:
  - `response = chat(messages, tools=TOOLS)`。
  - 履歴への追加は `add_assistant_message(messages, response)`(`Message` をそのまま渡せる。L26)。
  - 表示は `print(text_from_message(response))`。
  - 終了判定は `if response.stop_reason != "tool_use": return messages`。
  - 続きは `tool_results = run_tools(response)` と `add_user_message(messages, tool_results)`。
- **エラーが出たとき**: まず、エラーメッセージの最後の行(例外の種類と文)と、どのセルの何行目かを読む。原因が分からなければ、メッセージを貼ってほしい。